# Paper experiments — §5.3, §5.4, §6

**Use Colab for these** (GPU + Drive data). Run after the main eval cell finishes.

| Section | What | Time (T4) |
|---------|------|-----------|
| §5.4 | Codec sweep (`eval_codec.py`) | ~2–4 h |
| §5.3 | Train ASV19 → eval ASV21 | ~4 h train + ~30 min eval |
| §6.1 | 3 branch ablations | ~3×4 h each |
| §6.2 | 2 fusion ablations | ~2×4 h each |
| §6.3 | WavLM last-layer ablation | ~4 h |

Prerequisites: same setup as main notebook (Drive mounted, repo cloned, fixes applied).

In [ ]:
# Cell 0 — Shared setup (run once per session)
import os, re, glob, yaml, shutil, py_compile
from pathlib import Path

REPO = '/content/AI-Innovation'
DRIVE_DATA = '/content/drive/MyDrive/AI-Innovation-Data'
CFG = 'configs/gpu_local.yaml'

os.chdir(REPO)
os.environ['PYTHONPATH'] = REPO

with open(CFG) as f:
    cfg = yaml.safe_load(f)
cfg['data']['data_root'] = DRIVE_DATA
cfg['data']['num_workers'] = 0
with open(CFG, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

# rawnet + soundfile fixes
rn = Path('src/models/rawnet_branch.py')
rn.write_text(rn.read_text().replace('band_pass_center.unsqueeze(1)', 'band_pass_center'))

CKPT = 'outputs/ai_audio_detection/checkpoint_best.pt'
if not Path(CKPT).exists():
    found = glob.glob(f'{DRIVE_DATA}/checkpoints/**/checkpoint_best.pt', recursive=True)
    shutil.copy2(found[0], CKPT)
print('Setup OK')

In [ ]:
# §5.4 — Codec robustness (no retraining)
# Paste eval_codec.py if not in repo:
if not Path('scripts/eval_codec.py').exists():
    raise FileNotFoundError('Upload scripts/eval_codec.py from repo to Colab')

COMMON = f'--config {CFG} --checkpoint {CKPT}'
!python scripts/eval_codec.py {COMMON} --output-dir outputs/eval/codec_sweep
print('§5.4 done -> outputs/eval/codec_sweep/codec_metrics.json')

In [ ]:
# §5.3 — Cross-dataset: train on ASVspoof 2019 only, eval on 2021 test
!python -m src.training.train --config {CFG} \
    --sources asvspoof2019 \
    --experiment-name cross_ds_train19 \
    --batch_size 64 --num_workers 0

CROSS_CKPT = 'outputs/cross_ds_train19/checkpoint_best.pt'
!python scripts/evaluate.py --config {CFG} --checkpoint {CROSS_CKPT} \
    --sources asvspoof2021 \
    --output-dir outputs/eval/cross_19train_21test \
    --label 'Train19→Test21'
print('§5.3 done -> outputs/eval/cross_19train_21test/metrics.json')

In [ ]:
# §6.1 — Branch ablations (~4 h each on T4; run overnight)
import subprocess

ABLATIONS = [
    ('ablation_spectral_only', ['ssl', 'rawnet']),
    ('ablation_ssl_only',      ['spectral', 'rawnet']),
    ('ablation_rawnet_only',   ['spectral', 'ssl']),
]

for name, disabled in ABLATIONS:
    print('\n=== TRAIN', name, '===')
    cmd = [
        'python', '-m', 'src.training.train', '--config', CFG,
        '--experiment-name', name, '--batch_size', '64', '--num_workers', '0',
    ]
    for b in disabled:
        cmd += ['--disable-branch', b]
    subprocess.run(cmd, check=True)

    ckpt = f'outputs/{name}/checkpoint_best.pt'
    subprocess.run([
        'python', 'scripts/evaluate.py', '--config', CFG, '--checkpoint', ckpt,
        '--output-dir', f'outputs/eval/{name}', '--label', name,
    ], check=True)
print('§6.1 done')

In [ ]:
# §6.2 — Fusion ablations
for method in ['concat', 'average']:
    name = f'ablation_{method}_fusion'
    !python -m src.training.train --config {CFG} --fusion-method {method} --experiment-name {name} --batch_size 64 --num_workers 0
    !python scripts/evaluate.py --config {CFG} --checkpoint outputs/{name}/checkpoint_best.pt --output-dir outputs/eval/{name} --label {name}
print('§6.2 done')

In [ ]:
# §6.3 — WavLM last-layer-only ablation
!python -m src.training.train --config {CFG} --ssl-layer-mode last_layer \
    --experiment-name ablation_wavlm_last_layer --batch_size 64 --num_workers 0
!python scripts/evaluate.py --config {CFG} \
    --checkpoint outputs/ablation_wavlm_last_layer/checkpoint_best.pt \
    --output-dir outputs/eval/ablation_wavlm_last_layer \
    --label 'WavLM last layer'
print('§6.3 done')

In [ ]:
# §7.3 — Failure analysis (after full-model evaluate.py)
!python scripts/analyze_failures.py \
    --predictions outputs/eval/full_model/predictions.npz \
    --manifest data/metadata/master_manifest_segmented.csv \
    --output outputs/eval/full_model/failure_cases.csv
!head -30 outputs/eval/full_model/failure_cases.csv

In [ ]:
!zip -r paper_experiments_extra.zip outputs/eval outputs/ablation_* outputs/cross_ds_train19 2>/dev/null; ls -lh paper_experiments_extra.zip